# Imidazolone & Thiazolone Library Generation

Virtual combinatorial library of COX-2 selective inhibitors derived from 
commercially available aldehydes, carboxylic acids and amines via 
reactions *in silico*.

**Pipeline:** Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones  
**Parallel branch:** Oxazolones → Sulphur-Exchange → Thiazolones  
**Amines** feed into the Aminolysis-GFPc step as co-reactants.

All three building-block sets pass Price → Veber → Brenk+PAINS filters before reactions.  
Final products (Imidazolones and Thiazolones) are saved as **separate** CSV files.


In [1]:
from rdkit import Chem
import numpy as np
import pandas as pd

import py_utils as pu

print(f"py_utils v{pu.__version__}")

py_utils v3.4


## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂

In [2]:
sdf_path_aldehydes   = 'mol_files/0. EnamineSDFs/TESTER_Aldehydes_120.sdf'
sdf_path_carboxylics = 'mol_files/0. EnamineSDFs/TESTER_CarboxylicAcids_750.sdf'
sdf_path_amines      = 'mol_files/0. EnamineSDFs/TESTER_PrimaryAmines_680.sdf'

df_aldehydes_raw   = pu.sdf_to_dataframe(sdf_path_aldehydes,   id_prefix='A')
df_carboxylics_raw = pu.sdf_to_dataframe(sdf_path_carboxylics, id_prefix='C')
df_amines_raw      = pu.sdf_to_dataframe(sdf_path_amines,      id_prefix='N') 

pu.report_df_size(df_aldehydes_raw,   'Aldehydes - Raw')
pu.report_df_size(df_carboxylics_raw, 'Carboxylics - Raw')
pu.report_df_size(df_amines_raw,      'Amines - Raw')

[SDF Reader] Loaded 120 unique compounds from mol_files/0. EnamineSDFs/TESTER_Aldehydes_120.sdf
[SDF Reader] Loaded 750 unique compounds from mol_files/0. EnamineSDFs/TESTER_CarboxylicAcids_750.sdf
[SDF Reader] Loaded 680 unique compounds from mol_files/0. EnamineSDFs/TESTER_PrimaryAmines_680.sdf
[Aldehydes - Raw] 120 rows
[Carboxylics - Raw] 750 rows
[Amines - Raw] 680 rows


## 🔸 2. Price filtration

Query the Enamine Store API for real-time pricing.  Compounds exceeding the
price thresholds are discarded during retrieval (no wasted work).

In [3]:
client = pu.EnamineClient()

print('=== Querying Enamine API for Aldehydes ===')
df_aldehydes_cheap = pu.add_enamine_prices(df_aldehydes_raw, client=client)

print('\n=== Querying Enamine API for Carboxylic Acids ===')
df_carboxylics_cheap = pu.add_enamine_prices(df_carboxylics_raw, client=client)

print('\n=== Querying Enamine API for Amines ===')
df_amines_cheap = pu.add_enamine_prices(df_amines_raw, client=client)

print('\n' + '='*50)
print('PRICING SUMMARY')
print('='*50)
pu.report_df_size(df_aldehydes_cheap,   'Aldehydes - Cheap')
pu.report_df_size(df_carboxylics_cheap, 'Carboxylics - Cheap')
pu.report_df_size(df_amines_cheap,      'Amines - Cheap')

=== Querying Enamine API for Aldehydes ===
[Enamine Pricing] Loaded 413 valid + 1137 invalid cached
[Enamine Pricing] Processing 120 compounds...
[Enamine Pricing] Using cache for 120 compounds
[Enamine Pricing] Querying 0 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Completed: 36/120 with valid pricing
⚠️  Removed 84 compounds (no valid pricing)

=== Querying Enamine API for Carboxylic Acids ===
[Enamine Pricing] Loaded 413 valid + 1137 invalid cached
[Enamine Pricing] Processing 750 compounds...
[Enamine Pricing] Using cache for 750 compounds
[Enamine Pricing] Querying 0 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Completed: 145/750 with valid pricing
⚠️  Removed 605 compounds (no valid pricing)

=== Querying Enamine API for Amines ===
[Enamine Pricing] Loaded 413 valid + 1137 invalid cached
[Enamine Pricing] Processing 680 compounds...
[Enamine Pricing] Using cache for 680 compound

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply Veber bioavailability criteria.
Adjust `VEBER_BB` limits as needed.

In [4]:
VEBER_BB = dict(max_tPSA=90, max_RotB=10, max_LogP=3)

df_aldehydes_druglike, df_aldehydes_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_aldehydes_cheap), **VEBER_BB,
)
df_carboxylics_druglike, df_carboxylics_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_carboxylics_cheap), **VEBER_BB,
)
df_amines_druglike, df_amines_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_amines_cheap), **VEBER_BB,
)

pu.report_df_size(df_aldehydes_druglike,   'Aldehydes - Druglike')
pu.report_df_size(df_carboxylics_druglike, 'Carboxylics - Druglike')
pu.report_df_size(df_amines_druglike,      'Amines - Druglike')

# Save Veber-filtered building blocks
pu.save_dataframe_as_csv(df_aldehydes_druglike,   'mol_files/1. Building Blocks/Aldehydes_druglike')
pu.save_dataframe_as_csv(df_aldehydes_rej,        'mol_files/1. Building Blocks/Aldehydes_rejected_veber')
pu.save_dataframe_as_csv(df_carboxylics_druglike, 'mol_files/1. Building Blocks/Carboxylics_druglike')
pu.save_dataframe_as_csv(df_carboxylics_rej,      'mol_files/1. Building Blocks/Carboxylics_rejected_veber')
pu.save_dataframe_as_csv(df_amines_druglike,      'mol_files/1. Building Blocks/Amines_druglike')
pu.save_dataframe_as_csv(df_amines_rej,           'mol_files/1. Building Blocks/Amines_rejected_veber')

[filter_Veber] 29/36 accepted (80.6%), 7 rejected
[filter_Veber] 116/145 accepted (80.0%), 29 rejected
[Aldehydes - Druglike] 29 rows
[Carboxylics - Druglike] 116 rows
Saved 29 compounds to: mol_files/1. Building Blocks/Aldehydes_druglike_29cmpds.csv
Saved 7 compounds to: mol_files/1. Building Blocks/Aldehydes_rejected_veber_7cmpds.csv
Saved 116 compounds to: mol_files/1. Building Blocks/Carboxylics_druglike_116cmpds.csv
Saved 29 compounds to: mol_files/1. Building Blocks/Carboxylics_rejected_veber_29cmpds.csv


## 🔸 4. Brenk + PAINS filters

Remove compounds containing Brenk structural alerts or PAINS substructures.

In [5]:
df_aldehydes_untoxic,   df_aldehydes_rej   = pu.filter_brenk(df_aldehydes_druglike)
df_carboxylics_untoxic, df_carboxylics_rej = pu.filter_brenk(df_carboxylics_druglike)
df_amines_untoxic,      df_amines_rej      = pu.filter_brenk(df_amines_druglike)

pu.report_df_size(df_aldehydes_untoxic,   'Aldehydes - Untoxic')
pu.report_df_size(df_carboxylics_untoxic, 'Carboxylics - Untoxic')
pu.report_df_size(df_amines_untoxic,      'Amines - Untoxic')

# Save Brenk-filtered building blocks
pu.save_dataframe_as_csv(df_aldehydes_untoxic,   'mol_files/1. Building Blocks/Aldehydes_untoxic')
pu.save_dataframe_as_csv(df_aldehydes_rej,       'mol_files/1. Building Blocks/Aldehydes_rejected_brenk')
pu.save_dataframe_as_csv(df_carboxylics_untoxic, 'mol_files/1. Building Blocks/Carboxylics_untoxic')
pu.save_dataframe_as_csv(df_carboxylics_rej,     'mol_files/1. Building Blocks/Carboxylics_rejected_brenk')
pu.save_dataframe_as_csv(df_amines_untoxic,      'mol_files/1. Building Blocks/Amines_untoxic')
pu.save_dataframe_as_csv(df_amines_rej,          'mol_files/1. Building Blocks/Amines_rejected_brenk')

NotImplementedError: filter_brenk is not yet implemented. Provide Brenk and PAINS SMARTS patterns to proceed.

## 🔹 5. Erlenmeyer-Plöchl reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [ ]:
df_oxazolones_raw = pu.rxn_ErlenmeyerPlochl(df_aldehydes_untoxic, df_carboxylics_untoxic)

pu.report_df_size(df_oxazolones_raw, 'Oxazolones - Raw')

## 🔸 6. Veber filtering (of oxazolones)

Compute descriptors and apply Veber criteria to oxazolones.

In [ ]:
df_oxazolones_druglike, df_oxazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_oxazolones_raw),
    max_tPSA=90, max_RotB=10, max_LogP=3,  # TODO: adjust for intermediates if needed
)

pu.report_df_size(df_oxazolones_druglike, 'Oxazolones - Druglike')

# Save intermediates
pu.save_dataframe_as_csv(df_oxazolones_druglike, 'mol_files/2. Intermediates/Oxazolones')
pu.save_dataframe_as_csv(df_oxazolones_rej,      'mol_files/2. Intermediates/Oxazolones_rejected_veber')

## 🔹 7. Aminolysis-GFPc reaction

Oxazolones + Amines → Imidazolones

In [ ]:
df_imidazolones_raw = pu.rxn_AminolysisGFPc(df_oxazolones_druglike, df_amines_untoxic)

pu.report_df_size(df_imidazolones_raw, 'Imidazolones - Raw')

## 🔹 8. Sulphur-Exchange reaction

Oxazolones → Thiazolones

> **Note:** input is `df_oxazolones_druglike` (post-Veber, Cell 6) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [ ]:
df_thiazolones_raw = pu.rxn_SulphurExchange(df_oxazolones_druglike)

pu.report_df_size(df_thiazolones_raw, 'Thiazolones - Raw')

## 🔸 9. Veber filtering (of products)

Compute descriptors and apply Veber criteria to both product families.
Adjust `VEBER_PRODUCTS` limits independently from building blocks.

In [ ]:
VEBER_PRODUCTS = dict(max_tPSA=90, max_RotB=10, max_LogP=3)  # TODO: adjust limits for products

df_imidazolones_druglike, df_imidazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_imidazolones_raw), **VEBER_PRODUCTS,
)
df_thiazolones_druglike, df_thiazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_thiazolones_raw), **VEBER_PRODUCTS,
)

pu.report_df_size(df_imidazolones_druglike, 'Imidazolones - Druglike')
pu.report_df_size(df_thiazolones_druglike,  'Thiazolones - Druglike')

## 📤 10. Save CSVs

Imidazolones and Thiazolones saved as **separate** CSV files.

In [ ]:
# TODO: define columns
# df_imidazolones_druglike and df_thiazolones_druglike must be saved SEPARATELY
pu.save_dataframe_as_csv(df_imidazolones_druglike, 'mol_files/3. Products/Imidazolones')
pu.save_dataframe_as_csv(df_thiazolones_druglike,  'mol_files/3. Products/Thiazolones')